# Kruskal-Wallis H Test

Non-parametric test to determine whether there are statistically significant differences between groups.

## 1. Setup & Data Loading

In [11]:
import pandas as pd
import numpy as np
from scipy import stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_excel("../data/Nierenzell-Ca.xlsx")

print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst rows:")
df.head(10)

Shape: (185, 3)

Data types:
TPM       float64
Gruppe        str
Rang        int64
dtype: object

First rows:


,TPM,Gruppe,Rang
0,38676.048884,A,132
1,84091.070979,A,182
2,78645.108567,A,179
3,41349.520739,A,139
4,27086.218345,A,88
5,80360.182405,A,180
6,74323.818454,A,176
7,38963.579359,A,133
8,50071.976690,A,148
9,41902.330278,A,140


## 2. Variable Identification

In [12]:
group_col = "Gruppe"
dependent_col = "TPM"

print(f"Grouping variable: {group_col}")
print(f"Dependent variable: {dependent_col}")
print(f"\nUnique groups: {df[group_col].unique()}")
print(f"Number of groups: {df[group_col].nunique()}")
print(f"\nSample sizes per group:")
print(df[group_col].value_counts().sort_index())
print(f"\nDescriptive statistics per group:")
df.groupby(group_col)[dependent_col].describe()

Grouping variable: Gruppe
Dependent variable: TPM

Unique groups: <StringArray>
['A', 'B', 'C', 'D']
Length: 4, dtype: str
Number of groups: 4

Sample sizes per group:
Gruppe
A     23
B      3
C     19
D    140
Name: count, dtype: int64

Descriptive statistics per group:


,count,mean,std,min,25%,50%,75%,max
Gruppe,,,,,,,,
A,23.0,55373.521339,23957.650878,13797.034108,38819.814121,50982.460922,75448.382441,113119.528156
B,3.0,59882.317706,8051.915435,54102.882290,55283.812674,56464.743058,62772.035414,69079.327771
C,19.0,37540.413939,21993.816725,1621.155099,23650.422118,35023.646240,57075.616726,66276.786295
D,140.0,27014.495460,17565.992154,3658.541827,14240.392484,24653.411535,32762.836198,104610.924719


## 3. Assumption Checks

The Kruskal-Wallis test requires:
1. **Independent observations** — each observation belongs to only one group.
2. **Ordinal or continuous dependent variable.**
3. **Similar distribution shapes** across groups (not identical, but similar shape so that differences reflect location shifts).

In [ ]:
groups = df[group_col].unique()
n_groups = len(groups)

# Check 1: Independent observations
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")
print("=> Observations assumed independent (one measurement per subject per group).")

# Check 2: Dependent variable type
print(f"\nDependent variable '{dependent_col}' dtype: {df[dependent_col].dtype}")
print("=> Continuous variable — assumption satisfied.")

# Check 3: Similar distribution shapes — histogram overlay
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram overlay
for g in sorted(groups):
    subset = df.loc[df[group_col] == g, dependent_col].dropna()
    axes[0].hist(subset, alpha=0.5, label=f"{group_col} = {g}", bins="auto", edgecolor="black")
axes[0].set_xlabel(dependent_col)
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram Overlay by Group")
axes[0].legend()

# QQ plots per group
for g in sorted(groups):
    subset = df.loc[df[group_col] == g, dependent_col].dropna()
    stats.probplot(subset, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (all groups overlaid)")

plt.tight_layout()
fig.savefig("../figures/assumption_checks.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nVisual inspection: if histograms show roughly similar shapes (not necessarily")
print("normal), the assumption of similar distributions is reasonable.")

## 4. Kruskal-Wallis Test

In [14]:
group_data = [group[dependent_col].dropna().values for _, group in df.groupby(group_col)]

H_stat, p_value = stats.kruskal(*group_data)
dof = n_groups - 1

print("=" * 50)
print("       Kruskal-Wallis H Test Results")
print("=" * 50)
print(f"  H-statistic : {H_stat:.4f}")
print(f"  Degrees of freedom : {dof}")
print(f"  p-value     : {p_value:.6f}")
print("=" * 50)

alpha = 0.05
if p_value < alpha:
    print(f"\n=> Result is SIGNIFICANT at α = {alpha}.")
    print("   At least one group differs significantly from the others.")
else:
    print(f"\n=> Result is NOT significant at α = {alpha}.")
    print("   No significant differences detected between groups.")

       Kruskal-Wallis H Test Results
  H-statistic : 37.1032
  Degrees of freedom : 3
  p-value     : 0.000000

=> Result is SIGNIFICANT at α = 0.05.
   At least one group differs significantly from the others.


## 5. Post-Hoc Testing (Dunn's Test)

In [17]:
if p_value < alpha:
    print("Kruskal-Wallis was significant — running Dunn's test with Bonferroni correction.\n")
    dunn_results = sp.posthoc_dunn(df, val_col=dependent_col, group_col=group_col, p_adjust="bonferroni")
    print("Pairwise p-values (Bonferroni-corrected):")
    display(dunn_results.style.map(
        lambda v: "background-color: #ffcccc" if v < alpha else ""
    ))
else:
    print("Kruskal-Wallis was not significant — post-hoc testing is not warranted.")

Kruskal-Wallis was significant — running Dunn's test with Bonferroni correction.

Pairwise p-values (Bonferroni-corrected):


,A,B,C,D
A,1.000000,1.000000,0.187476,0.000000
B,1.000000,1.000000,0.598102,0.042489
C,0.187476,0.598102,1.000000,0.149153
D,0.000000,0.042489,0.149153,1.000000


## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot
sns.boxplot(data=df, x=group_col, y=dependent_col, ax=axes[0], palette="Set2")
axes[0].set_title(f"Box Plot of {dependent_col} by {group_col}")
axes[0].set_xlabel(group_col)
axes[0].set_ylabel(dependent_col)

# Violin plot
sns.violinplot(data=df, x=group_col, y=dependent_col, ax=axes[1], palette="Set2", inner="quartile")
axes[1].set_title(f"Violin Plot of {dependent_col} by {group_col}")
axes[1].set_xlabel(group_col)
axes[1].set_ylabel(dependent_col)

fig.suptitle(f"Distribution of {dependent_col} across {group_col} Groups", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig("../figures/box_violin_plots.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Summary

**Interpretation:**

A Kruskal-Wallis H test was conducted to determine whether there are statistically significant differences in **TPM** across the levels of **Gruppe**.

- If the p-value is **below 0.05**, we conclude that at least one group's distribution of TPM differs significantly from the others. Dunn's post-hoc test (with Bonferroni correction) identifies which specific group pairs differ.
- If the p-value is **above 0.05**, we fail to reject the null hypothesis — there is no sufficient evidence that TPM distributions differ across groups.

The Kruskal-Wallis test is appropriate here because it does not assume normality, making it suitable for skewed data or data with outliers. The test compares rank-transformed values across groups rather than raw means.